# Визуализация инсайтов модели (EDA & XAI)
Запускать ячейки можно по отдельности с помощью зеленого треугольника слева.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from stage_2.data import build_panel, DatasetConfig, ROOT

# Настройка красивого стиля для всех графиков
sns.set_theme(style="whitegrid", context="talk")
OUT_DIR = ROOT / "stage_2" / "output"


In [ ]:
# Здесь мы исправили warning от seaborn, добавив hue
print("Генерация графиков Feature Importance...")
xgb_path = OUT_DIR / "xgb_feature_importance_gain.csv"
cat_path = OUT_DIR / "catboost_feature_importance.csv"

df_xgb = pd.read_csv(xgb_path).head(10)
df_cat = pd.read_csv(cat_path).head(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=df_xgb, x="gain", y="feature_name", ax=axes[0], hue="feature_name", legend=False, palette="viridis")
axes[0].set_title("Топ-10 признаков: XGBoost", fontsize=14, pad=15)
axes[0].set_xlabel("Относительная важность (Gain)")
axes[0].set_ylabel("")

sns.barplot(data=df_cat, x="importance", y="feature", ax=axes[1], hue="feature", legend=False, palette="mako")
axes[1].set_title("Топ-10 признаков: CatBoost", fontsize=14, pad=15)
axes[1].set_xlabel("Относительная важность (%)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show() # В PyCharm график появится прямо в окне Scientific Mode


In [ ]:
print("Сборка данных для матрицы...")
cfg = DatasetConfig()
panel, feature_cols, target = build_panel(cfg)

top_features = [
    "months_since_last_incident", "incidents_prev_3m", "incident_rate_prev_3m",
    "lifetime_incidents_cnt", "lifetime_incident_rate", "months_in_company",
    "trips_prev_3m", "mpg_drop_ratio", "truck_churn_rate", "detention_minutes_avg",
    "average_mpg", "idle_spike_ratio", "on_time_delivery_rate", "trips_completed", target
]
top_features = [f for f in top_features if f in panel.columns]

df_corr = panel[top_features].corr(method='spearman')

plt.figure(figsize=(18, 16))
mask = np.triu(np.ones_like(df_corr, dtype=bool))
sns.heatmap(
    df_corr, mask=mask, cmap="coolwarm", annot=True,
    fmt=".2f", vmin=-1, vmax=1, square=True, linewidths=.5, cbar_kws={"shrink": .8}
)
plt.title("Матрица ранговой корреляции (Spearman)", fontsize=16, pad=20)
plt.tight_layout()
plt.show()


In [ ]:
print("Генерация бизнес-графиков...")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 4.1 Дисбаланс классов (Target imbalance)
# Показываем, почему задача сложная
target_counts = panel[target].value_counts()
sns.barplot(x=target_counts.index, y=target_counts.values, ax=axes[0], hue=target_counts.index, legend=False, palette={0: "#2ecc71", 1: "#e74c3c"})
axes[0].set_title("Дисбаланс классов (Target)", fontsize=14, pad=15)
axes[0].set_xlabel("Инцидент (0 - Нет, 1 - Есть)")
axes[0].set_ylabel("Количество месяцев (driver-month)")

# 4.2 Как самый важный признак влияет на риск (Boxplot)
# Сравниваем месяцы без инцидентов и с инцидентами по признаку "Свежесть прошлого инцидента"
sns.boxplot(data=panel, x=target, y="months_since_last_incident", ax=axes[1], hue=target, legend=False, palette={0: "#2ecc71", 1: "#e74c3c"})
axes[1].set_title("Влияние давности прошлого инцидента на риск", fontsize=14, pad=15)
axes[1].set_xlabel("Случился ли инцидент в этом месяце? (Target)")
axes[1].set_ylabel("Месяцев с прошлого инцидента")

plt.tight_layout()
plt.show()